In [2]:
import pandas as pd
BASE = "C:/Users/HP/Downloads/DS/da9/walmart-demand-forecast"
PROCESSED = f"{BASE}/data/processed"

final_all = pd.read_csv(f"{PROCESSED}/production_forecast_ALL_final.csv")

print("=== KIỂM TRA TOÀN VẸN FILE CUỐI CÙNG ===")
print(f"Tổng dòng: {len(final_all)}, tổng cặp: {final_all[['Store','Dept']].drop_duplicates().shape[0]}")
print(f"Giá trị null trong forecast: {final_all['forecast'].isna().sum()}")
print(f"Giá trị null trong lower_95/upper_95: {final_all['lower_95'].isna().sum()}/{final_all['upper_95'].isna().sum()}")
print(f"\nForecast âm còn sót: {(final_all['forecast']<0).sum()}")

print("\n=== TỔNG DOANH SỐ DỰ BÁO 4 TUẦN TỚI THEO PHƯƠNG PHÁP ===")
print(final_all.groupby('method')['forecast'].sum())

print(f"\nTổng dự báo TOÀN CÔNG TY 4 tuần tới: {final_all['forecast'].sum():,.0f} USD")
print(f"Trung bình mỗi tuần: {final_all['forecast'].sum()/4:,.0f} USD")
print(f"(Để so sánh: doanh số trung bình lịch sử mỗi tuần ~47.1 triệu USD)")

=== KIỂM TRA TOÀN VẸN FILE CUỐI CÙNG ===
Tổng dòng: 13324, tổng cặp: 3331
Giá trị null trong forecast: 0
Giá trị null trong lower_95/upper_95: 0/0

Forecast âm còn sót: 0

=== TỔNG DOANH SỐ DỰ BÁO 4 TUẦN TỚI THEO PHƯƠNG PHÁP ===
method
LightGBM          6.997836e+05
SARIMAX           2.052963e+08
Seasonal_Naive    1.196259e+04
Name: forecast, dtype: float64

Tổng dự báo TOÀN CÔNG TY 4 tuần tới: 206,008,088 USD
Trung bình mỗi tuần: 51,502,022 USD
(Để so sánh: doanh số trung bình lịch sử mỗi tuần ~47.1 triệu USD)


In [3]:
final_all['Date'] = pd.to_datetime(final_all['Date'])
by_week = final_all.groupby('Date')['forecast'].sum().sort_index()
print("=== TỔNG DỰ BÁO THEO TỪNG TUẦN (kiểm tra có phải do Thanksgiving) ===")
print(by_week)
print(f"\nCác ngày dự báo: {sorted(final_all['Date'].unique())}")
print("Thanksgiving 2012: 2012-11-22 (rơi vào tuần kết thúc 2012-11-23)")

=== TỔNG DỰ BÁO THEO TỪNG TUẦN (kiểm tra có phải do Thanksgiving) ===
Date
2012-08-31    1.262919e+01
2012-09-07    2.163823e+01
2012-09-14    4.432575e+01
2012-09-21    5.366157e+01
2012-09-28    1.236245e+01
2012-10-05    1.587300e+01
2012-10-12    7.057241e+00
2012-10-19    1.038696e+02
2012-10-26    3.082660e+02
2012-11-02    4.803153e+07
2012-11-09    4.817832e+07
2012-11-16    4.662809e+07
2012-11-23    6.316957e+07
Name: forecast, dtype: float64

Các ngày dự báo: [Timestamp('2012-08-31 00:00:00'), Timestamp('2012-09-07 00:00:00'), Timestamp('2012-09-14 00:00:00'), Timestamp('2012-09-21 00:00:00'), Timestamp('2012-09-28 00:00:00'), Timestamp('2012-10-05 00:00:00'), Timestamp('2012-10-12 00:00:00'), Timestamp('2012-10-19 00:00:00'), Timestamp('2012-10-26 00:00:00'), Timestamp('2012-11-02 00:00:00'), Timestamp('2012-11-09 00:00:00'), Timestamp('2012-11-16 00:00:00'), Timestamp('2012-11-23 00:00:00')]
Thanksgiving 2012: 2012-11-22 (rơi vào tuần kết thúc 2012-11-23)


In [4]:
#check nguồn
print("=== KIỂM TRA NGUỒN GỐC 9 TUẦN BẤT THƯỜNG ===")
weird_dates = final_all[final_all['Date'] < '2012-11-01']
print(weird_dates['method'].value_counts())
print(weird_dates.groupby(['method'])[['Store','Dept']].apply(lambda g: g.drop_duplicates().shape[0]))
print(weird_dates.head(15))

=== KIỂM TRA NGUỒN GỐC 9 TUẦN BẤT THƯỜNG ===
method
SARIMAX    27
Name: count, dtype: int64
method
SARIMAX    15
dtype: int64
      Store  Dept       Date   forecast  lower_95    upper_95   method
2084     30    17 2012-08-31  12.629194       0.0  222.067597  SARIMAX
2085     30    17 2012-09-07  21.638232       0.0  202.688196  SARIMAX
2086     30    17 2012-09-14  29.107137       0.0  238.545540  SARIMAX
2087     30    17 2012-09-21  34.257287       0.0  215.307251  SARIMAX
2904     23    45 2012-09-14  15.218614       0.0   73.547343  SARIMAX
2905     23    45 2012-09-21  19.404278       0.0   77.733007  SARIMAX
2906     23    45 2012-09-28  12.362448       0.0   70.691177  SARIMAX
2907     23    45 2012-10-05  15.873000       0.0   74.201729  SARIMAX
3716     18    54 2012-10-26  41.838223       0.0  172.445184  SARIMAX
4112     44    59 2012-10-12   7.057241       0.0   75.637924  SARIMAX
4113     44    59 2012-10-19  10.176238       0.0   74.106936  SARIMAX
4114     44    59 2012

In [5]:
import duckdb
con = duckdb.connect()

check_pairs = [(30,17),(23,45),(18,54),(44,59),(29,54),(38,85)]
for s, d in check_pairs:
    last_actual = con.execute(f"""
        SELECT MAX(Date) FROM read_parquet('{PROCESSED}/walmart_clean.parquet')
        WHERE Store={s} AND Dept={d}
    """).fetchone()[0]
    print(f"Store={s}, Dept={d}: ngày cuối có dữ liệu THẬT = {last_actual}")

Store=30, Dept=17: ngày cuối có dữ liệu THẬT = 2012-08-24
Store=23, Dept=45: ngày cuối có dữ liệu THẬT = 2012-09-07
Store=18, Dept=54: ngày cuối có dữ liệu THẬT = 2012-10-19
Store=44, Dept=59: ngày cuối có dữ liệu THẬT = 2012-10-05
Store=29, Dept=54: ngày cuối có dữ liệu THẬT = 2012-10-12
Store=38, Dept=85: ngày cuối có dữ liệu THẬT = 2012-10-19


In [9]:
import duckdb
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX

PROCESSED = f"{BASE}/data/processed"
con = duckdb.connect()

# Lấy khung ngày TOÀN CỤC 1 lần 
global_dates = con.execute(f"""
    SELECT DISTINCT Date FROM read_parquet('{PROCESSED}/walmart_clean.parquet') ORDER BY Date
""").df()
global_dates['Date'] = pd.to_datetime(global_dates['Date'])
full_date_index = pd.date_range(global_dates['Date'].min(), global_dates['Date'].max(), freq='W-FRI')

broken_pairs = final_all[(final_all['Date'] < '2012-11-01') & (final_all['method']=='SARIMAX')][['Store','Dept']].drop_duplicates()
print(f"Số cặp cần fit lại: {len(broken_pairs)}")

fixed_forecasts = []
for _, row in broken_pairs.iterrows():
    s, d = int(row['Store']), int(row['Dept'])
    series_df = con.execute(f"""
        SELECT Date, Weekly_Sales FROM read_parquet('{PROCESSED}/walmart_clean.parquet')
        WHERE Store={s} AND Dept={d} ORDER BY Date
    """).df()
    series_df['Date'] = pd.to_datetime(series_df['Date'])

    
    ts_s = series_df.set_index('Date')['Weekly_Sales'].reindex(full_date_index)
    print(f"Store={s}, Dept={d}: ngày cuối THẬT trong data = {series_df['Date'].max()}, "
          f"sau reindex kết thúc = {ts_s.index.max()}, số NaN cuối chuỗi = {ts_s.iloc[-10:].isna().sum()}")

    ts_s = ts_s.interpolate(limit=10, limit_direction='both')

    if ts_s.isna().sum() > 0:
        print(f"  CẢNH BÁO: vẫn còn {ts_s.isna().sum()} NaN sau interpolate - bỏ qua cặp này, cần xử lý riêng")
        continue

    model = SARIMAX(ts_s, order=(2,0,2), seasonal_order=(1,0,0,52),
                     enforce_stationarity=False, enforce_invertibility=False)
    fitted = model.fit(disp=False, maxiter=50)
    residuals = fitted.resid[55:] if len(fitted.resid) > 75 else fitted.resid[-20:]
    point_forecast = fitted.get_forecast(steps=4).predicted_mean

    np.random.seed(42)
    boot = np.array([point_forecast.values + np.random.choice(residuals, 4, replace=True) for _ in range(2000)])
    lower_95 = np.maximum(0, np.percentile(boot, 2.5, axis=0))
    upper_95 = np.maximum(0, np.percentile(boot, 97.5, axis=0))

    fixed_forecasts.append(pd.DataFrame({
        'Store': s, 'Dept': d, 'Date': point_forecast.index,
        'forecast': np.maximum(0, point_forecast.values),
        'lower_95': lower_95, 'upper_95': upper_95, 'method': 'SARIMAX'
    }))

fixed_df = pd.concat(fixed_forecasts, ignore_index=True)
print(f"\n=== Đã fix xong {fixed_df[['Store','Dept']].drop_duplicates().shape[0]} cặp ===")
print(f"Ngày forecast sau khi fix: {sorted(fixed_df['Date'].unique())}")

# Loại bỏ dòng lỗi cũ 
final_all['pair_key'] = final_all['Store'].astype(str) + '_' + final_all['Dept'].astype(str)
broken_keys = broken_pairs['Store'].astype(str) + '_' + broken_pairs['Dept'].astype(str)
final_all_clean = final_all[~final_all['pair_key'].isin(broken_keys)].drop(columns=['pair_key'])

final_all_clean = pd.concat([final_all_clean, fixed_df], ignore_index=True)
final_all_clean.to_csv(f"{PROCESSED}/production_forecast_ALL_final_v2.csv", index=False)

print(f"\nTổng dòng sau khi sửa: {len(final_all_clean)}")
print(final_all_clean.groupby('Date')['forecast'].sum())

Số cặp cần fit lại: 15
Store=30, Dept=17: ngày cuối THẬT trong data = 2012-08-24 00:00:00, sau reindex kết thúc = 2012-10-26 00:00:00, số NaN cuối chuỗi = 9
Store=23, Dept=45: ngày cuối THẬT trong data = 2012-09-07 00:00:00, sau reindex kết thúc = 2012-10-26 00:00:00, số NaN cuối chuỗi = 8


C:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Store=18, Dept=54: ngày cuối THẬT trong data = 2012-10-19 00:00:00, sau reindex kết thúc = 2012-10-26 00:00:00, số NaN cuối chuỗi = 4


C:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Store=44, Dept=59: ngày cuối THẬT trong data = 2012-10-05 00:00:00, sau reindex kết thúc = 2012-10-26 00:00:00, số NaN cuối chuỗi = 5
Store=29, Dept=54: ngày cuối THẬT trong data = 2012-10-12 00:00:00, sau reindex kết thúc = 2012-10-26 00:00:00, số NaN cuối chuỗi = 3
Store=38, Dept=85: ngày cuối THẬT trong data = 2012-10-19 00:00:00, sau reindex kết thúc = 2012-10-26 00:00:00, số NaN cuối chuỗi = 1


...

Store=16, Dept=54: ngày cuối THẬT trong data = 2012-10-12 00:00:00, sau reindex kết thúc = 2012-10-26 00:00:00, số NaN cuối chuỗi = 2
Store=7, Dept=54: ngày cuối THẬT trong data = 2012-10-19 00:00:00, sau reindex kết thúc = 2012-10-26 00:00:00, số NaN cuối chuỗi = 2


C:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Store=25, Dept=54: ngày cuối THẬT trong data = 2012-10-12 00:00:00, sau reindex kết thúc = 2012-10-26 00:00:00, số NaN cuối chuỗi = 5
Store=21, Dept=54: ngày cuối THẬT trong data = 2012-10-19 00:00:00, sau reindex kết thúc = 2012-10-26 00:00:00, số NaN cuối chuỗi = 4


C:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Store=37, Dept=6: ngày cuối THẬT trong data = 2012-10-19 00:00:00, sau reindex kết thúc = 2012-10-26 00:00:00, số NaN cuối chuỗi = 3

=== Đã fix xong 15 cặp ===
Ngày forecast sau khi fix: [Timestamp('2012-11-02 00:00:00'), Timestamp('2012-11-09 00:00:00'), Timestamp('2012-11-16 00:00:00'), Timestamp('2012-11-23 00:00:00')]

Tổng dòng sau khi sửa: 13324
Date
2012-11-02    4.803155e+07
2012-11-09    4.817836e+07
2012-11-16    4.662821e+07
2012-11-23    6.316993e+07
Name: forecast, dtype: float64
